Lab Assignemnt- 10
Devansh Rathaur
1024170380


Design a Cognitive Assistant for Reading Labels of medicine bottle (You can download
Dataset from Internet)
**bold text**

In [ ]:


!pip -q install easyocr gradio opencv-python-headless pandas

import re
import cv2
import easyocr
import numpy as np
import pandas as pd
import gradio as gr

# ================================================================
# 1. INITIALIZE OCR MODEL
# ================================================================

print("Loading EasyOCR model...")

reader = easyocr.Reader(['en'], gpu=False)

print("OCR model loaded successfully!")


# ================================================================
# 2. MEDICINE REFERENCE DATASET
# ================================================================

# Small demonstration dataset.
# You can expand it using a publicly available medicine dataset.

medicine_data = [
    {
        "name": "Paracetamol",
        "ingredient": "Paracetamol",
        "strength": "500 mg",
        "form": "Tablet"
    },
    {
        "name": "Ibuprofen",
        "ingredient": "Ibuprofen",
        "strength": "200 mg",
        "form": "Tablet"
    },
    {
        "name": "Cetirizine",
        "ingredient": "Cetirizine",
        "strength": "10 mg",
        "form": "Tablet"
    },
    {
        "name": "Amoxicillin",
        "ingredient": "Amoxicillin",
        "strength": "500 mg",
        "form": "Capsule"
    },
    {
        "name": "Azithromycin",
        "ingredient": "Azithromycin",
        "strength": "500 mg",
        "form": "Tablet"
    }
]

medicine_df = pd.DataFrame(medicine_data)

print("\nMedicine reference dataset:")
print(medicine_df)


# ================================================================
# 3. IMAGE PREPROCESSING USING OPENCV
# ================================================================

def preprocess_image(image):

    # Convert RGB to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Resize image for better OCR
    height, width = gray.shape

    if width < 1200:
        scale = 1200 / width
        gray = cv2.resize(
            gray,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv2.INTER_CUBIC
        )

    # Noise removal
    denoised = cv2.bilateralFilter(
        gray,
        9,
        75,
        75
    )

    # Contrast enhancement
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    enhanced = clahe.apply(denoised)

    # Adaptive thresholding
    processed = cv2.adaptiveThreshold(
        enhanced,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        11
    )

    return processed


# ================================================================
# 4. OCR TEXT EXTRACTION
# ================================================================

def extract_text(image):

    processed_image = preprocess_image(image)

    # Extract text using EasyOCR
    results = reader.readtext(
        processed_image,
        detail=1,
        paragraph=False
    )

    extracted_text = []

    for result in results:

        bounding_box = result[0]
        text = result[1]
        confidence = result[2]

        if confidence > 0.15:

            extracted_text.append(text)

    return "\n".join(extracted_text)


# ================================================================
# 5. NLP AND INFORMATION EXTRACTION
# ================================================================

def extract_information(text):

    text_upper = text.upper()

    information = {}

    # Medicine name candidate
    lines = [
        line.strip()
        for line in text.split("\n")
        if line.strip()
    ]

    metadata_words = [
        "BATCH", "LOT", "EXP", "MFG",
        "MANUFACTURED", "MANUFACTURER",
        "STORE", "STORAGE", "TABLET",
        "CAPSULE", "SYRUP", "WARNING",
        "COMPOSITION"
    ]

    candidate = "Not detected"

    for line in lines:

        if (
            len(line) >= 3
            and not any(word in line.upper()
                        for word in metadata_words)
            and not re.search(r"\d{2,}", line)
        ):
            candidate = line
            break

    information["Medicine Name (OCR Candidate)"] = candidate

    # Active ingredient
    ingredient = "Not detected"

    for name in medicine_df["ingredient"]:

        if name.upper() in text_upper:
            ingredient = name
            break

    information["Active Ingredient (Dataset Match)"] = ingredient

    # Strength extraction
    strength = re.findall(
        r"\b\d+(?:\.\d+)?\s*(?:MG|MCG|G|ML|IU|%|MG/ML)\b",
        text_upper
    )

    information["Strength"] = (
        ", ".join(dict.fromkeys(strength))
        if strength else "Not detected"
    )

    # Dosage form
    forms = [
        "TABLET", "CAPSULE", "SYRUP",
        "SUSPENSION", "INJECTION",
        "CREAM", "OINTMENT", "DROPS"
    ]

    detected_forms = [
        form for form in forms
        if form in text_upper
    ]

    information["Dosage Form"] = (
        ", ".join(detected_forms)
        if detected_forms else "Not detected"
    )

    # Batch number
    batch = re.search(
        r"(?:BATCH|B\.?NO|LOT)\s*[:.#-]?\s*([A-Z0-9/-]+)",
        text_upper
    )

    information["Batch Number"] = (
        batch.group(1) if batch else "Not detected"
    )

    # Expiry date
    expiry = re.search(
        r"(?:EXP|EXPIRY|EXPIRES)\.?\s*[:.-]?\s*"
        r"(\d{1,2}[/-]\d{2,4}|\d{2,4}[/-]\d{1,2}|"
        r"[A-Z]{3,9}\s*\d{4})",
        text_upper
    )

    information["Expiry Date"] = (
        expiry.group(1) if expiry else "Not detected"
    )

    # Manufacturing date
    mfg = re.search(
        r"(?:MFG|MFD|MANUFACTURED)\.?\s*[:.-]?\s*"
        r"(\d{1,2}[/-]\d{2,4}|\d{2,4}[/-]\d{1,2}|"
        r"[A-Z]{3,9}\s*\d{4})",
        text_upper
    )

    information["Manufacturing Date"] = (
        mfg.group(1) if mfg else "Not detected"
    )

    # Manufacturer
    manufacturer = re.search(
        r"(?:MFG\s*BY|MANUFACTURED\s*BY|"
        r"MARKETED\s*BY|MANUFACTURER)\s*[:.-]?\s*(.+)",
        text,
        re.IGNORECASE
    )

    information["Manufacturer"] = (
        manufacturer.group(1).strip()
        if manufacturer else "Not detected"
    )

    # Storage instructions
    storage = re.search(
        r"(STORE.{0,100})",
        text,
        re.IGNORECASE
    )

    information["Storage Instructions"] = (
        storage.group(1).strip()
        if storage else "Not detected"
    )

    return information


# ================================================================
# 6. MEDICINE INFORMATION LOOKUP
# ================================================================

def lookup_medicine(information):

    ingredient = information[
        "Active Ingredient (Dataset Match)"
    ]

    if ingredient == "Not detected":

        return (
            "Medicine not found in the local reference dataset. "
            "Please verify the label manually."
        )

    result = medicine_df[
        medicine_df["ingredient"] == ingredient
    ]

    if result.empty:

        return "No matching medicine record found."

    medicine = result.iloc[0]

    return (
        f"Reference Medicine: {medicine['name']}\n"
        f"Active Ingredient: {medicine['ingredient']}\n"
        f"Reference Strength: {medicine['strength']}\n"
        f"Dosage Form: {medicine['form']}\n\n"
        "This is a dataset match, not independent verification "
        "of the uploaded medicine."
    )


# ================================================================
# 7. COGNITIVE ASSISTANT - COMPLETE PIPELINE
# ================================================================

def medicine_assistant(image):

    if image is None:

        return (
            "Please upload a medicine bottle image.",
            {},
            ""
        )

    # Step 1: OCR
    text = extract_text(image)

    if not text.strip():

        return (
            "No readable text detected. Try a clearer image.",
            {},
            ""
        )

    # Step 2: NLP and information extraction
    information = extract_information(text)

    # Step 3: Knowledge retrieval
    medicine_info = lookup_medicine(information)

    # Step 4: Response generation
    response = (
        "MEDICINE LABEL READING ASSISTANT\n\n"
        + medicine_info
        + "\n\nEXTRACTED INFORMATION:\n"
    )

    for key, value in information.items():

        response += f"\n{key}: {value}"

    response += (
        "\n\nSAFETY NOTICE:\n"
        "OCR can misread medicine names, strengths and dates. "
        "Verify every extracted field against the original "
        "packaging and consult a pharmacist. Do not use this "
        "system to determine dosage or decide whether to take "
        "a medicine."
    )

    return response, information, text


interface = gr.Interface(

    fn=medicine_assistant,

    inputs=gr.Image(
        type="numpy",
        label="Upload Medicine Bottle Image"
    ),

    outputs=[
        gr.Textbox(
            label="Medicine Assistant Response",
            lines=18
        ),

        gr.JSON(
            label="Extracted Medicine Information"
        ),

        gr.Textbox(
            label="Raw OCR Extracted Text",
            lines=10
        )
    ],

    title="Medicine Bottle Label Reading Cognitive Assistant",

    description=(
        "Upload a photograph of a medicine bottle. "
        "The assistant uses OpenCV and EasyOCR to read "
        "the label and extract medicine information."
    ),

    allow_flagging="never"
)




print("\nLaunching Medicine Label Reading Assistant...")

interface.launch(
    share=True,
    debug=True
)


Design an Emotion-aware Cognitive Assistant using Facial Expression ***Detection***

In [ ]:

# ================================================================
# EMOTION-AWARE COGNITIVE ASSISTANT
# Facial Expression Detection using DeepFace
# Python + OpenCV + Deep Learning + Gradio
# ================================================================

# 1. INSTALL LIBRARIES

!pip -q install deepface gradio opencv-python-headless

import cv2
import numpy as np
import gradio as gr
import pandas as pd

from deepface import DeepFace


# ================================================================
# 2. EMOTION CATEGORIES
# ================================================================

emotions = [
    "happy",
    "sad",
    "angry",
    "fear",
    "surprise",
    "disgust",
    "neutral"
]

print("Supported facial expression categories:")
print(emotions)


# ================================================================
# 3. KNOWLEDGE BASE
# ================================================================

emotion_knowledge_base = {

    "happy": {
        "description": "A positive or smiling facial expression.",
        "response": (
            "You appear to have a positive facial expression! "
            "Keep enjoying the moment. Would you like to share "
            "what is making you smile?"
        ),
        "suggestion": "Continue doing activities you enjoy."
    },

    "sad": {
        "description": "A facial expression commonly associated with sadness.",
        "response": (
            "Your facial expression appears somewhat sad. "
            "If something is bothering you, you can talk about it. "
            "Would you like to share what's on your mind?"
        ),
        "suggestion": "Consider talking to someone you trust."
    },

    "angry": {
        "description": "A facial expression that may indicate frustration.",
        "response": (
            "Your facial expression appears tense. "
            "If you are feeling frustrated, consider taking "
            "a short break. I am here if you want to talk."
        ),
        "suggestion": "Try a short break or slow breathing."
    },

    "fear": {
        "description": "A facial expression that may indicate fear or concern.",
        "response": (
            "Your facial expression may indicate concern. "
            "Take a moment to pause. If something is worrying "
            "you, you can share it with me."
        ),
        "suggestion": "Try a gentle grounding exercise."
    },

    "surprise": {
        "description": "A facial expression commonly associated with surprise.",
        "response": (
            "You seem to have a surprised facial expression! "
            "Did something unexpected happen?"
        ),
        "suggestion": "Take a moment to understand what surprised you."
    },

    "disgust": {
        "description": "A facial expression commonly associated with disgust.",
        "response": (
            "Your facial expression appears to show displeasure. "
            "Is something bothering you?"
        ),
        "suggestion": "Take a moment away from anything unpleasant."
    },

    "neutral": {
        "description": "A relatively neutral facial expression.",
        "response": (
            "Your facial expression appears neutral. "
            "How is your day going? I am ready to assist you."
        ),
        "suggestion": "Continue with your daily activities."
    }
}


# ================================================================
# 4. COGNITIVE REASONING RULES
# ================================================================

CONFIDENCE_THRESHOLD = 40


def cognitive_reasoning(emotion, confidence):

    # RULE 1: Low confidence handling

    if confidence < CONFIDENCE_THRESHOLD:

        return {
            "status": "Uncertain prediction",
            "response": (
                "I could not confidently classify the visible "
                "facial expression. Please try another image "
                "with better lighting."
            ),
            "suggestion": "Try uploading a clearer image."
        }

    # RULE 2: Emotion-based response generation

    result = emotion_knowledge_base.get(emotion)

    if result is None:

        return {
            "status": "Unknown expression",
            "response": "Unable to classify the facial expression.",
            "suggestion": "Try another image."
        }

    return {
        "status": "Expression classified",
        "response": result["response"],
        "suggestion": result["suggestion"]
    }


# ================================================================
# 5. IMAGE PROCESSING AND FACE DETECTION
# ================================================================

def detect_faces(image):

    # Convert RGB to BGR for OpenCV / DeepFace

    image_bgr = cv2.cvtColor(
        image,
        cv2.COLOR_RGB2BGR
    )

    # Detect facial expressions using pretrained DeepFace model

    results = DeepFace.analyze(
        img_path=image_bgr,
        actions=["emotion"],
        detector_backend="opencv",
        enforce_detection=False,
        silent=True
    )

    # DeepFace can return a dictionary or list of dictionaries

    if isinstance(results, dict):
        results = [results]

    return results, image_bgr


# ================================================================
# 6. EMOTION DETECTION
# ================================================================

def analyze_emotion(image):

    if image is None:

        return None, "Please upload an image.", {}

    try:

        # Face detection and expression recognition

        results, image_bgr = detect_faces(image)

        output_image = image_bgr.copy()

        all_results = []

        # Process every detected face

        for index, face in enumerate(results):

            emotion_scores = face.get("emotion", {})

            if not emotion_scores:
                continue

            # Get dominant facial expression

            emotion = max(
                emotion_scores,
                key=emotion_scores.get
            ).lower()

            confidence = float(
                emotion_scores[emotion]
            )

            # Cognitive reasoning

            reasoning = cognitive_reasoning(
                emotion,
                confidence
            )

            # Bounding box

            region = face.get("region", {})

            x = int(region.get("x", 0))
            y = int(region.get("y", 0))
            w = int(region.get("w", 0))
            h = int(region.get("h", 0))

            # Draw rectangle around face

            if w > 0 and h > 0:

                cv2.rectangle(
                    output_image,
                    (x, y),
                    (x + w, y + h),
                    (0, 255, 0),
                    2
                )

                label = f"{emotion}: {confidence:.1f}%"

                cv2.putText(
                    output_image,
                    label,
                    (x, max(y - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 255, 0),
                    2
                )

            # Store expression results

            all_results.append({

                "Face": index + 1,

                "Detected Expression": emotion,

                "Confidence": round(confidence, 2),

                "Expression Description":
                    emotion_knowledge_base[emotion]["description"],

                "Assistant Response":
                    reasoning["response"],

                "Suggestion":
                    reasoning["suggestion"],

                "Status":
                    reasoning["status"],

                "Expression Scores": {
                    key: round(float(value), 2)
                    for key, value in emotion_scores.items()
                }

            })

        # No valid face result

        if not all_results:

            return (
                image,
                "No usable facial expression was detected. "
                "Please upload a clearer photograph.",
                {}
            )

        # Generate final assistant response

        final_response = ""

        for result in all_results:

            final_response += (
                f"\nFACE {result['Face']}\n"
                f"Detected Expression: "
                f"{result['Detected Expression']}\n"
                f"Confidence: {result['Confidence']}%\n"
                f"Status: {result['Status']}\n\n"
                f"Assistant: {result['Assistant Response']}\n\n"
                f"Suggestion: {result['Suggestion']}\n"
                f"{'-' * 45}\n"
            )

        # Convert output image back to RGB

        output_image = cv2.cvtColor(
            output_image,
            cv2.COLOR_BGR2RGB
        )

        return (
            output_image,
            final_response,
            all_results
        )

    except Exception as e:

        return (
            image,
            f"Error processing image: {str(e)}",
            {}
        )


# ================================================================
# 7. INTERACTIVE GRADIO INTERFACE
# ================================================================

demo = gr.Interface(

    fn=analyze_emotion,

    inputs=gr.Image(
        type="numpy",
        label="Upload Your Face Image"
    ),

    outputs=[

        gr.Image(
            label="Face Detection Result"
        ),

        gr.Textbox(
            label="Cognitive Assistant Response",
            lines=15
        ),

        gr.JSON(
            label="Detailed Expression Analysis"
        )

    ],

    title="Emotion-Aware Cognitive Assistant",

    description=(
        "Upload a facial image to detect visible facial "
        "expressions using a pretrained DeepFace model. "
        "The assistant generates supportive responses "
        "based on the estimated expression."
    ),

    examples=None,

    allow_flagging="never"

)


# ================================================================
# 8. LAUNCH APPLICATION
# ================================================================

print("Starting Emotion-Aware Cognitive Assistant...")

demo.launch(
    share=True,
    debug=True
)
